# GT vs Pred 3D Overlay

This notebook loads ground-truth SWC trees and predicted trees from a validation pickle, then overlays a selected pair in an interactive Plotly 3D view.

Controls:
- Toggle GT/pred nodes on/off
- Adjust per-tree opacity


In [1]:
from pathlib import Path

# Update these paths to your data
GT_DIR = Path('path/to/gt_swc_dir')
PRED_PKL = Path('path/to/validation.pkl')
EMA_KEY = None  # e.g., 'ema_0.999'

# Selection settings
MATCH_BY_SIZE = True
PAIR_INDEX = 0
GT_INDEX = 0
PRED_INDEX = 0

# Initial display settings
SHOW_GT_NODES = True
SHOW_PRED_NODES = True
GT_OPACITY = 0.9
PRED_OPACITY = 0.7
GT_EDGE_COLOR = '#9ecae1'
PRED_EDGE_COLOR = '#f0a6b5'


In [2]:
import importlib.util
import sys
from pathlib import Path

CWD = Path('.').resolve()
if (CWD / 'validation' / '3D_plot.py').exists():
    REPO_ROOT = CWD
elif (CWD / '3D_plot.py').exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SCRIPT_PATH = REPO_ROOT / 'validation' / '3D_plot.py'
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'Could not find helper script at {SCRIPT_PATH}')

spec = importlib.util.spec_from_file_location('plot3d_helper', SCRIPT_PATH)
plot3d = importlib.util.module_from_spec(spec)
if spec and spec.loader:
    spec.loader.exec_module(plot3d)
else:
    raise ImportError(f'Could not load helper script at {SCRIPT_PATH}')


In [3]:
# Load graphs
gt_files, gt_graphs, pred_graphs = plot3d.load_graphs(GT_DIR, PRED_PKL, EMA_KEY)

if MATCH_BY_SIZE:
    pairs = plot3d._match_by_size(gt_graphs, pred_graphs)
    if not pairs:
        raise RuntimeError('No GT/pred pairs found to plot.')
    if PAIR_INDEX < 0 or PAIR_INDEX >= len(pairs):
        raise IndexError(f'PAIR_INDEX {PAIR_INDEX} out of range (0..{len(pairs)-1}).')
    pair = pairs[PAIR_INDEX]
    gt_idx = pair['gt_idx']
    pred_idx = pair['pred_idx']
    match_note = f"{pair['match_type']} (size diff {pair['size_diff']})"
else:
    gt_idx = GT_INDEX
    pred_idx = PRED_INDEX
    match_note = 'manual'

gt = gt_graphs[gt_idx]
pred = pred_graphs[pred_idx]
gt_name = gt_files[gt_idx].name if gt_idx < len(gt_files) else f'gt_{gt_idx}'
title = (
    f'GT {gt_idx} ({gt_name}, n={gt.number_of_nodes()}) ' 
    f'vs Pred {pred_idx} (n={pred.number_of_nodes()}) | {match_note}'
)

fig, trace_indices = plot3d.build_overlay_figure(
    gt,
    pred,
    show_gt_nodes=SHOW_GT_NODES,
    show_pred_nodes=SHOW_PRED_NODES,
    gt_opacity=GT_OPACITY,
    pred_opacity=PRED_OPACITY,
    gt_edge_color=GT_EDGE_COLOR,
    pred_edge_color=PRED_EDGE_COLOR,
    title=title,
    as_widget=True,
)

plot3d.attach_widgets(
    fig,
    trace_indices,
    gt_label='GT',
    pred_label='Pred',
    initial_gt_opacity=GT_OPACITY,
    initial_pred_opacity=PRED_OPACITY,
    show_gt_nodes=SHOW_GT_NODES,
    show_pred_nodes=SHOW_PRED_NODES,
)


NotADirectoryError: Provided path is not a directory: path/to/gt_swc_dir